# Module 5.2: Fast Fourier Transform (FFT) & Windowing
### 🇵🇱 Szybka Transformata Fouriera (FFT) i Okienkowanie

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m05_l02_fft_and_windowing.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Calculate magnitude spectra using numpy.fft, prevent spectral leakage using Hanning windows, and calibrate frequency bins.  
> 🇵🇱 **Streszczenie:** Obliczaj widmo amplitudowe w numpy.fft, eliminuj przeciek widmowy oknem Hanninga i kalibruj osie częstotliwości.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Time Domain vs Frequency Spectrum**: Microscopic bearing damage is invisible in time signals but exhibits distinct spectral peaks.
2. **Spectral Leakage & Hanning Windowing**: Non-periodic signal truncation causes energy smear; windowing restores frequency resolution.
3. **Characteristic Defect Frequencies**: Outer race defect (BPFO) peaks at $f_{BPFO} = \frac{N_{balls}}{2} \cdot f_{shaft} \cdot \left(1 - \frac{d}{D}\cos\alpha\right)$.


In [ ]:
# 1. Synthesize Accelerometer Vibration with Bearing Defect (BPFO)
np.random.seed(42)
Fs = 2000.0  # Sampling frequency: 2000 Hz
T = 1.0 / Fs
N = 2048
time_axis = np.linspace(0.0, N*T, N, endpoint=False)

f_shaft = 30.0            # 30 Hz shaft rotational speed (1800 RPM)
f_bpfo = 3.58 * f_shaft   # 107.4 Hz outer race defect

sig_1x = 1.8 * np.sin(2.0 * np.pi * f_shaft * time_axis)
sig_bpfo = 1.4 * np.sin(2.0 * np.pi * f_bpfo * time_axis)
noise = 0.5 * np.random.randn(N)

vibration_accel = sig_1x + sig_bpfo + noise
print(f"✓ Vibration signal synthesized: {N} samples over {N*T:.2f} seconds.")


In [ ]:
# 2. Compute FFT Spectrum with Hanning Window
window = np.hanning(N)
windowed_sig = vibration_accel * window

fft_vals = np.fft.rfft(windowed_sig)
fft_freqs = np.fft.rfftfreq(N, T)
fft_mag = (2.0 / N) * np.abs(fft_vals)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(time_axis[:300], vibration_accel[:300], color='#0284c7', linewidth=1.2)
axes[0].set_title("Time Domain: Accelerometer Raw Vibration", fontweight='bold')
axes[0].set_ylabel("Accel [m/s²]")
axes[0].grid(True, alpha=0.3)

axes[1].plot(fft_freqs, fft_mag, color='#10b981', linewidth=1.5, label='FFT Magnitude Spectrum')
axes[1].axvline(f_shaft, color='#f59e0b', linestyle='--', label=f'1X Shaft ({f_shaft:.1f} Hz)')
axes[1].axvline(f_bpfo, color='#ef4444', linestyle='-', linewidth=2, label=f'BPFO Defect ({f_bpfo:.1f} Hz)')
axes[1].set_xlim(0, 300)
axes[1].set_title("Frequency Domain: Bearing Fault Signature Exposed!", fontweight='bold')
axes[1].set_xlabel("Frequency [Hz]")
axes[1].set_ylabel("Amplitude Peak [m/s²]")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Compute discrete FFT using numpy.fft.rfft *(🇵🇱 Wylicz dyskretną transformatę FFT za pomocą numpy.fft.rfft)*
- [ ] **Task 2:** Construct the exact frequency axis vector using np.fft.rfftfreq *(🇵🇱 Zbuduj oś częstotliwości za pomocą np.fft.rfftfreq)*
- [ ] **Task 3:** Apply a Hanning window to eliminate sharp boundary discontinuities *(🇵🇱 Zastosuj okno Hanninga, aby zlikwidować nieciągłości na brzegach okna)*


In [ ]:
# Exercise Challenge:
# Locate the frequency with maximum magnitude between 90 Hz and 130 Hz (BPFO range)
bpfo_band = (fft_freqs >= 90) & (fft_freqs <= 130)
peak_freq = fft_freqs[bpfo_band][np.argmax(fft_mag[bpfo_band])]
print(f"Detected Bearing Defect Frequency Peak: {peak_freq:.2f} Hz (Expected ~107.4 Hz)")
assert abs(peak_freq - f_bpfo) < 2.0, "FFT defect peak detection out of tolerance!"
print("✓ Vibration analysis exercise passed.")
